In [1]:
import json
import shutil
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm
# from ultralytics import YOLO
# import mediapipe as mp
from concurrent.futures import ThreadPoolExecutor, as_completed
# from turbojpeg import TurboJPEG
import re
import torch
import os
import argparse

In [2]:
def iter_groups_from_jsonl(jsonl_path: Path, image_root: Path, max_per_prompt=None):
    """
    Yields:
        (group_name, lazy_items_function)
    where lazy_items_function returns [(Path, score, group_id), ...] when called
    """

    def resolve_src(r):
        raw = Path(r["image_path"])
        if raw.is_absolute():
            return raw
        return image_root / raw.parent.parent / f"{r['group_id']}_images" / raw.name

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            prompt = re.sub(r'[<>:"/\\|?*]', '_', obj["prompt"]).strip().replace(" ", "_")
            results = obj.get("results", [])

            if max_per_prompt:
                results = results[:max_per_prompt]

            if not results:
                continue

            # Return a lambda that will resolve paths only when called
            def make_items_loader(results_copy):
                def load_items():
                    items = []
                    for r in results_copy:
                        p = resolve_src(r)
                        items.append((p, r["score"], r["group_id"]))
                    return items
                return load_items

            yield prompt, make_items_loader(results)


In [4]:
jsonl_1 = r"G:\Thesis\ImageRetrieval\Professions_125k_ISCO_Aligned\retrieval_results_batchsize_10.jsonl"

jsonl_2 = r"E:\ImageRetrieval\Professions_125k_Cleaned\125k_retrieval_results_batchsize_10.jsonl"

image_root = r"F:\Thesis"

In [5]:
groups = iter_groups_from_jsonl(jsonl_1, image_root)
for f in groups:
    print(f[0])
    print(f[0].split('_')[-1].upper())
    break

A_photo_of_an_accountant
ACCOUNTANT


In [6]:
groups_2 = iter_groups_from_jsonl(jsonl_2, image_root)
for f in groups_2:
    print(f[0].split('_')[-1].upper())
    break

ACCOUNTANT


In [5]:
def extract_occupation(prompt: str) -> str:
    """
    Extracts full job title from standardized prompt formats:
        Male_{JobTitle}
        Female_{JobTitle}
        A_photo_of_an_{JobTitle}
        A_photo_of_a_{JobTitle}

    Returns:
        Job title in UPPER_CASE with underscores preserved.
    """
    prompt = prompt.strip()

    prefixes = (
        "Male_",
        "Female_",
        "A_photo_of_an_",
        "A_photo_of_a_",
    )

    for p in prefixes:
        if prompt.startswith(p):
            return prompt[len(p):].upper()

    raise ValueError(f"Unrecognized prompt format: {prompt}")

# --- Build lookup set from jsonl_2 (single pass) ---
occupations_2 = set()

for group_name, _ in iter_groups_from_jsonl(jsonl_2, image_root):
    occupations_2.add(extract_occupation(group_name))

print(f"Loaded {len(occupations_2)} occupations from jsonl_2")

NameError: name 'jsonl_2' is not defined

In [ ]:
# --- Find common occupations (single pass) ---
common_occupations = set()

for group_name, _ in iter_groups_from_jsonl(jsonl_1, image_root):
    occupation_1 = extract_occupation(group_name)

    if occupation_1 in occupations_2:
        print(f"--> Found Common Occupation: {occupation_1}")
        common_occupations.add(occupation_1)


print(f"Common occupations: {len(common_occupations)}")

Loaded 201 occupations from jsonl_2
--> Found Common Occupation: ACCOUNTANT
--> Found Common Occupation: ACTOR
--> Found Common Occupation: ACTUARY
--> Found Common Occupation: ADMINISTRATIVE_ASSISTANT
--> Found Common Occupation: ADMINISTRATOR
--> Found Common Occupation: AIR_TRAFFIC_CONTROLLER
--> Found Common Occupation: ANIMAL_TRAINER
--> Found Common Occupation: ANTHROPOLOGIST
--> Found Common Occupation: APPRAISER
--> Found Common Occupation: ARCHAEOLOGIST
--> Found Common Occupation: ARCHITECT
--> Found Common Occupation: ARCHIVIST
--> Found Common Occupation: ART_DIRECTOR
--> Found Common Occupation: ARTIST
--> Found Common Occupation: ASTRONAUT
--> Found Common Occupation: ATHLETE
--> Found Common Occupation: AUDIO_TECHNICIAN
--> Found Common Occupation: AUDITOR
--> Found Common Occupation: AUTOMOTIVE_DESIGNER
--> Found Common Occupation: BAKER
--> Found Common Occupation: BANKER
--> Found Common Occupation: BANKRUPTCY_SPECIALIST
--> Found Common Occupation: BARBER
--> Found C

In [10]:
from collections import defaultdict
from pathlib import Path


def build_occupation_index(jsonl_path: Path, image_root: Path):
    """
    Returns:
        Dict[str, Set[Path]]
        Mapping: OCCUPATION -> set of image paths
    """
    index = defaultdict(set)

    for group_name, load_items in iter_groups_from_jsonl(jsonl_path, image_root):
        print(group_name)
        occupation = extract_occupation(group_name)

        # Resolve lazily only when needed
        items = load_items()

        for img_path, score, group_id in items:
            index[occupation].add(img_path)

    return index


# ----------------------------
# Build indices
# ----------------------------

index_1 = build_occupation_index(jsonl_1, image_root)
index_2 = build_occupation_index(jsonl_2, image_root)

print(f"jsonl_1 occupations: {len(index_1)}")
print(f"jsonl_2 occupations: {len(index_2)}")


# ----------------------------
# Compute common occupation files
# ----------------------------

common_files_by_occupation = {}

common_occupations = set(index_1.keys()) & set(index_2.keys())

for occupation in sorted(common_occupations):
    files_1 = index_1[occupation]
    files_2 = index_2[occupation]

    common_files = files_1 & files_2   # set intersection

    if common_files:
        common_files_by_occupation[occupation] = common_files
        print(f"{occupation}: {len(common_files)} common files")


print(f"\nTotal common occupations: {len(common_files_by_occupation)}")


A_photo_of_an_accountant
A_photo_of_an_actor
A_photo_of_an_actuary
A_photo_of_an_administrative_assistant
A_photo_of_an_administrator
A_photo_of_an_air_traffic_controller
A_photo_of_an_animal_trainer
A_photo_of_an_anthropologist
A_photo_of_an_appraiser
A_photo_of_an_archaeologist
A_photo_of_an_architect
A_photo_of_an_archivist
A_photo_of_an_art_director
A_photo_of_an_artist
A_photo_of_an_astronaut
A_photo_of_an_astronomer
A_photo_of_an_athlete
A_photo_of_an_audio_technician
A_photo_of_an_auditor
A_photo_of_an_automotive_designer
A_photo_of_a_baker
A_photo_of_a_banker
A_photo_of_a_bankruptcy_specialist
A_photo_of_a_barber
A_photo_of_a_barista
A_photo_of_a_bartender
A_photo_of_a_basketball_player
A_photo_of_a_biologist
A_photo_of_a_biomedical_engineer
A_photo_of_a_blacksmith
A_photo_of_a_bodyguard
A_photo_of_a_bounty_hunter
A_photo_of_a_boxer
A_photo_of_a_brand_manager
A_photo_of_a_brewer
A_photo_of_a_bricklayer
A_photo_of_a_broker
A_photo_of_a_builder
A_photo_of_a_butcher
A_photo_of_a_c

In [12]:
from collections import defaultdict
from pathlib import Path

def build_filename_index(common_files_by_occupation: dict[str, set[Path]]):
    """
    Returns:
        Dict[str, Dict[str, Set[Path]]]
        OCCUPATION -> { filename -> set(Path) }
    """
    index = {}

    for occupation, paths in common_files_by_occupation.items():
        name_map = defaultdict(set)
        for p in paths:
            name_map[p.name].add(p)
        index[occupation] = name_map

    return index

filename_index = build_filename_index(common_files_by_occupation)

In [13]:
from pathlib import Path

def parse_valid_filename(line: str) -> str:
    """
    Parses:
        {score}_{group_id}_{image_name}.jpg
    Returns:
        image_name.jpg
    """
    line = line.strip()
    if not line:
        return None

    parts = line.split("_", 2)
    if len(parts) != 3:
        raise ValueError(f"Invalid valid.txt line: {line}")

    return parts[2]   # image_name.jpg

from collections import defaultdict

# def collect_valid_common_files(
#     jsonl_2: Path,
#     image_root: Path,
#     dir_variable: Path,
#     common_files_by_occupation: dict[str, set[Path]],
# ):
#     """
#     Returns:
#         Dict[str, Set[Path]]
#         OCCUPATION -> set of valid image paths that also exist in the
#         common_files_by_occupation intersection.
#     """
#     results = defaultdict(set)

#     for group_name, _ in iter_groups_from_jsonl(jsonl_2, image_root):
#         occupation = extract_occupation(group_name)

#         # Skip occupations that are not common
#         if occupation not in common_files_by_occupation:
#             continue

#         # Path to valid.txt for this prompt
#         prompt_dir = dir_variable / group_name
#         valid_path = prompt_dir / "valid.txt"

#         if not valid_path.exists():
#             print(f"[WARN] Missing valid.txt: {valid_path}")
#             continue

#         allowed_paths = common_files_by_occupation[occupation]

#         with open(valid_path, "r", encoding="utf-8") as f:
#             for line in f:
#                 img_name = parse_valid_filename(line)
#                 if img_name is None:
#                     continue

#                 # We must reconstruct the full image path so that it matches
#                 # the Path objects stored in common_files_by_occupation.
#                 #
#                 # If your valid images live under the same tree as iter_groups,
#                 # you typically only need the filename match.
#                 #
#                 # Safer approach: match by filename.
#                 matching = [
#                     p for p in allowed_paths
#                     if p.name == img_name
#                 ]

#                 for p in matching:
#                     results[occupation].add(p)

#     return results


from collections import defaultdict
from pathlib import Path


def collect_valid_common_files(
    jsonl_2: Path,
    image_root: Path,
    dir_variable: Path,
    common_files_by_occupation: dict[str, set[Path]],
):
    """
    Returns:
        Dict[str, Set[Path]]

    Semantics:
        - If an occupation has at least one valid.txt file → filter normally.
        - If an occupation has zero valid.txt files → occupation maps to empty set.
    """
    results = {occ: set() for occ in common_files_by_occupation}
    seen_valid_file = {occ: False for occ in common_files_by_occupation}

    for group_name, _ in iter_groups_from_jsonl(jsonl_2, image_root):
        occupation = extract_occupation(group_name)

        # Skip non-common occupations
        if occupation not in results:
            continue

        prompt_dir = dir_variable / group_name
        valid_path = prompt_dir / "valid.txt"

        if not valid_path.exists():
            continue

        seen_valid_file[occupation] = True
        allowed_paths = common_files_by_occupation[occupation]

        with open(valid_path, "r", encoding="utf-8") as f:
            for line in f:
                img_name = parse_valid_filename(line)
                if not img_name:
                    continue

                # Fast filename lookup (see index below)
                for p in filename_index[occupation].get(img_name, ()):
                    results[occupation].add(p)

    # Enforce rule: if no valid.txt existed → empty set
    for occ, seen in seen_valid_file.items():
        if not seen:
            results[occ].clear()

    return results


dir_variable = Path(r"E:\ImageRetrieval\Professions_125k_Cleaned")

valid_common_files = collect_valid_common_files(
    jsonl_2=jsonl_2,
    image_root=image_root,
    dir_variable=dir_variable,
    common_files_by_occupation=common_files_by_occupation,
)

for occupation, files in valid_common_files.items():
    print(f"{occupation}: {len(files)} valid common files")


ACCOUNTANT: 19022 valid common files
ACTOR: 47500 valid common files
ACTUARY: 22656 valid common files
ADMINISTRATIVE_ASSISTANT: 10175 valid common files
ADMINISTRATOR: 27035 valid common files
AIR_TRAFFIC_CONTROLLER: 20544 valid common files
ANIMAL_TRAINER: 24426 valid common files
ANTHROPOLOGIST: 24895 valid common files
APPRAISER: 35516 valid common files
ARCHAEOLOGIST: 16540 valid common files
ARCHITECT: 13513 valid common files
ARCHIVIST: 30689 valid common files
ARTIST: 21035 valid common files
ART_DIRECTOR: 37396 valid common files
ASTRONAUT: 12014 valid common files
ATHLETE: 23574 valid common files
AUDIO_TECHNICIAN: 23425 valid common files
AUDITOR: 29978 valid common files
AUTOMOTIVE_DESIGNER: 6510 valid common files
BAKER: 15821 valid common files
BANKER: 30654 valid common files
BANKRUPTCY_SPECIALIST: 25742 valid common files
BARBER: 22825 valid common files
BARISTA: 28634 valid common files
BARTENDER: 25988 valid common files
BIOLOGIST: 23089 valid common files
BIOMEDICAL_

In [ ]:
common_occupations = []
groups = iter_groups_from_jsonl(jsonl_1, image_root)
for f1 in groups:
    occupation_1 = f1[0].split('_')[-1].upper()
    print(f"Processing {occupation_1}")
    groups_2 = iter_groups_from_jsonl(jsonl_2, image_root)
    for f2 in groups_2:
        occupation_2 = f2[0].split('_')[-1].upper()
        
        if occupation_1 == occupation_2:
            print(f"--> Found Common Occupation: {occupation_1}")
            common_occupations += occupation_1
            break

common_occupations = set(common_occupations)
print(len(common_occupations))

Processing ACCOUNTANT
--> Found Common Occupation: ACCOUNTANT
Processing ACTOR
--> Found Common Occupation: ACTOR
Processing ACTUARY
--> Found Common Occupation: ACTUARY
Processing ASSISTANT
--> Found Common Occupation: ASSISTANT
Processing ADMINISTRATOR
--> Found Common Occupation: ADMINISTRATOR
Processing CONTROLLER
--> Found Common Occupation: CONTROLLER
Processing TRAINER
--> Found Common Occupation: TRAINER
Processing ANTHROPOLOGIST
--> Found Common Occupation: ANTHROPOLOGIST
Processing APPRAISER
--> Found Common Occupation: APPRAISER
Processing ARCHAEOLOGIST
--> Found Common Occupation: ARCHAEOLOGIST
Processing ARCHITECT
--> Found Common Occupation: ARCHITECT
Processing ARCHIVIST
--> Found Common Occupation: ARCHIVIST
Processing DIRECTOR
--> Found Common Occupation: DIRECTOR
Processing ARTIST
--> Found Common Occupation: ARTIST
Processing ASTRONAUT
--> Found Common Occupation: ASTRONAUT
Processing ASTRONOMER
Processing ATHLETE
--> Found Common Occupation: ATHLETE
Processing TECHN